# Data Pipeline

**Inputs** (in `data/`):
- `data/Spotify_dataset_gigasheet.csv` — raw Spotify audio-feature dataset
- `data/playlist.json` — raw Million Playlist Dataset slice (1,000 playlists)

**Outputs** (written to `outputs/`):
- `outputs/spotify_cleaned.csv` — cleaned song catalogue
- `outputs/USETHIS_output_filtered.csv` — playlist rows (pid, artist, track)
- `outputs/taste_profiles.csv` — per-user mean of 7 audio features (805 users)
- `outputs/user_liked_songs.csv` — per-user liked-song list joined with the catalogue

K-means cluster centres are written to `archive/K-means/cluster{5,10,15,20}.csv` for reference.

# 1. Data Cleaning


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
df = pd.read_csv("./data/Spotify_dataset_gigasheet.csv")
df

In [ ]:
print((df['Title'] == 'None').sum())
df.isna().sum()

In [ ]:
df['Album_type'].value_counts()

In [ ]:
#remove compilations
df = df[(df['Album_type'] != "compilation")].copy()

In [ ]:
df['Duration_min'].sort_values()

In [ ]:
# remove tracks longer than 10 mins
df = df[(df['Duration_min'] < 10)].copy()

In [ ]:
df = df.sort_values('Stream', ascending=False)
df.duplicated(subset=['Artist', 'Track']).value_counts()

In [ ]:
# remove duplicate artist/track combos
df = df.drop_duplicates(subset=['Artist', 'Track'], keep='first')

In [ ]:
numerical_cols = df.select_dtypes(include='number').columns
zero_counts = (df[numerical_cols] == 0).sum()
print(zero_counts)

In [ ]:
#remove rows with zero tempo and duration
df = df[(df['Tempo'] != 0) & (df['Duration_min'] != 0)].copy()

# set flag for zero views and streams
df['zero_engagement_flag'] = (df['Views'] == 0) | (df['Stream'] == 0)
df

In [ ]:
df.to_csv('outputs/spotify_cleaned.csv', index=False)

# 2. Playlist Extraction

Ports `archive/Playlist Cleaning.ipynb`:

1. **Unique-track whitelist** from the cleaned catalogue (archive cell 0 — originally read `cleandata.csv` and exported `artist_track.csv`; we build it from `spotify_cleaned.csv` produced in §1).
2. **Flatten `playlist.json`** into `(pid, artist_name, track_name)` rows (archive cell 1).
3. **Filter** the flattened extract against the whitelist (archive cell 2 — originally filtered against a separate `unique_tracks.csv`).

The filter here is redundant with the inner-join in §4 (both drop plays whose tracks are not in the catalogue), but it is kept for provenance and produces the same final `taste_profiles.csv` / `user_liked_songs.csv`.

In [ ]:
import json
import csv
import pandas as pd

# Step 1 (archive cell 0): build unique (artist, track) whitelist from the cleaned catalogue
catalogue = pd.read_csv('outputs/spotify_cleaned.csv', usecols=['Artist', 'Track'])
unique_tracks = set(zip(catalogue['Artist'].astype(str).str.strip(),
                        catalogue['Track'].astype(str).str.strip()))
print(f'Unique-track whitelist: {len(unique_tracks)} (artist, track) pairs')

# Step 2 (archive cell 1): flatten playlist.json -> (pid, artist_name, track_name)
with open('data/playlist.json') as f:
    data = json.load(f)

raw_rows = [
    {'pid': p['pid'], 'artist_name': t['artist_name'], 'track_name': t['track_name']}
    for p in data['playlists'] for t in p['tracks']
]
playlists_raw = pd.DataFrame(raw_rows)
print(f'Extracted {len(playlists_raw)} track-plays from {playlists_raw["pid"].nunique()} playlists '
      f'(avg {playlists_raw.groupby("pid").size().mean():.2f} songs per playlist)')

# Step 3 (archive cell 2): keep only plays whose (artist, track) is in the whitelist
mask = [(a.strip(), t.strip()) in unique_tracks
        for a, t in zip(playlists_raw['artist_name'], playlists_raw['track_name'])]
playlists_filtered = playlists_raw[mask].reset_index(drop=True)
removed = len(playlists_raw) - len(playlists_filtered)
print(f'Kept {len(playlists_filtered)} rows, removed {removed} '
      f'({removed/len(playlists_raw)*100:.1f}%) not in catalogue')

playlists_filtered.to_csv('outputs/USETHIS_output_filtered.csv', index=False)
print('Saved outputs/USETHIS_output_filtered.csv')

# 3. K-means Clustering

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv("outputs/spotify_cleaned.csv")
df
df.sort_values(by='Duration_min')
df['Album_type'].value_counts()

In [ ]:
# 7 taste profile features only — aligned with independence_tests.ipynb results
# Dropped: Loudness (redundant with Energy, MI=0.643), Liveness (weak taste signal),
#          Duration_min (not taste), EnergyLiveness (engineered/redundant)
features = [
    'Danceability', 'Energy', 'Valence', 'Acousticness',
    'Instrumentalness', 'Speechiness', 'Tempo'
]
X = df[features]
X

In [ ]:
# scale each column to have mean 0 and sd = 1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42)
df['cluster5'] = kmeans.fit_predict(X_scaled)

centers = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=features)
print(centers)
centers.to_csv('./archive/K-means/cluster5.csv', index=False)

In [ ]:
kmeans = KMeans(n_clusters=10, random_state=42)
df['cluster10'] = kmeans.fit_predict(X_scaled)

centers = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=features)
print(centers)
centers.to_csv('./archive/K-means/cluster10.csv', index=False)

In [ ]:
kmeans = KMeans(n_clusters=15, random_state=42)
df['cluster15'] = kmeans.fit_predict(X_scaled)

centers = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=features)
print(centers)
centers.to_csv('./archive/K-means/cluster15.csv', index=False)

In [ ]:
kmeans = KMeans(n_clusters=20, random_state=42)
df['cluster20'] = kmeans.fit_predict(X_scaled)

centers = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=features)
print(centers)
centers.to_csv('./archive/K-means/cluster20.csv', index=False)

# 4. Taste Profiles and Liked Songs

Builds a per-user taste profile by:
1. Joining playlist data (§2) with song audio features (§1)
2. Filtering playlists with fewer than 3 songs
3. Normalising Tempo to [0, 1]
4. Averaging the 7 taste features per user (`pid`)

Also exports `user_liked_songs.csv` — the per-user track list used by §3 for reward-function validation.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load datasets
songs = pd.read_csv('outputs/spotify_cleaned.csv')
playlists = pd.read_csv('outputs/USETHIS_output_filtered.csv')

print("=" * 60)
print("STEP 0: LOAD DATA")
print("=" * 60)
print(f"Songs dataset:    {songs.shape[0]} rows, {songs.shape[1]} columns")
print(f"Playlist dataset: {playlists.shape[0]} rows, {playlists.shape[1]} columns")
print(f"Unique playlists: {playlists['pid'].nunique()}")
print(f"Unique songs in playlists: {playlists.groupby(['artist_name','track_name']).ngroups}")
print(f"\nPlaylist columns: {list(playlists.columns)}")
print(f"Songs columns: {list(songs.columns)}")

In [ ]:
TASTE_FEATURES = [
    'Danceability', 'Energy', 'Valence',
    'Acousticness', 'Instrumentalness', 'Speechiness',
    'Tempo'
]

print("=" * 60)
print("STEP 1: DEDUPLICATE PLAYLISTS")
print("=" * 60)
print(f"Before deduplication: {len(playlists)} rows")

# Find duplicates before removing
dupes = playlists[playlists.duplicated(subset=['pid', 'artist_name', 'track_name'], keep='first')]
print(f"Duplicate rows found: {len(dupes)}")
print(f"\nExamples of duplicates removed:")
dupes_sample = dupes.head(10)
for _, row in dupes_sample.iterrows():
    print(f"  pid={row['pid']:>4d}  {row['artist_name']} - {row['track_name']}")

# Remove duplicates
playlists_clean = playlists.drop_duplicates(subset=['pid', 'artist_name', 'track_name'], keep='first')
print(f"\nAfter deduplication: {len(playlists_clean)} rows")
print(f"Rows removed: {len(playlists) - len(playlists_clean)}")

In [ ]:
print("=" * 60)
print("STEP 2: JOIN WITH SONG FEATURES")
print("=" * 60)

# Join on artist + track
joined = playlists_clean.merge(
    songs[['Artist', 'Track'] + TASTE_FEATURES],
    left_on=['artist_name', 'track_name'],
    right_on=['Artist', 'Track'],
    how='left'
)

matched = joined['Artist'].notna().sum()
unmatched = joined['Artist'].isna().sum()

print(f"Matched rows:   {matched} ({matched/len(joined)*100:.2f}%)")
print(f"Unmatched rows: {unmatched} ({unmatched/len(joined)*100:.2f}%)")

if unmatched > 0:
    print(f"\nUnmatched songs:")
    unmatched_songs = joined[joined['Artist'].isna()][['pid', 'artist_name', 'track_name']]
    for _, row in unmatched_songs.iterrows():
        print(f"  pid={row['pid']:>4d}  {row['artist_name']} - {row['track_name']}")

# Drop unmatched rows and extra columns
joined = joined.dropna(subset=['Artist'])
joined = joined.drop(columns=['Artist', 'Track'])

print(f"\nJoined dataset: {len(joined)} rows with {len(TASTE_FEATURES)} taste features")
print(f"Playlists in joined data: {joined['pid'].nunique()}")

In [ ]:
print("=" * 60)
print("STEP 3: FILTER SMALL PLAYLISTS (min 3 songs)")
print("=" * 60)

songs_per_playlist = joined.groupby('pid').size()

print("Playlist size distribution before filtering:")
print(f"  1-2 songs:  {(songs_per_playlist <= 2).sum()} playlists")
print(f"  3-5 songs:  {((songs_per_playlist >= 3) & (songs_per_playlist <= 5)).sum()} playlists")
print(f"  6-10 songs: {((songs_per_playlist >= 6) & (songs_per_playlist <= 10)).sum()} playlists")
print(f"  11+ songs:  {(songs_per_playlist >= 11).sum()} playlists")

# Filter
valid_pids = songs_per_playlist[songs_per_playlist >= 3].index
removed_pids = songs_per_playlist[songs_per_playlist < 3].index

print(f"\nPlaylists removed (< 3 songs): {len(removed_pids)}")
print(f"Playlists remaining: {len(valid_pids)}")

joined_filtered = joined[joined['pid'].isin(valid_pids)]
print(f"Rows remaining: {len(joined_filtered)}")

In [ ]:
print("=" * 60)
print("STEP 4: NORMALIZE TEMPO TO [0, 1]")
print("=" * 60)

tempo_min = joined_filtered['Tempo'].min()
tempo_max = joined_filtered['Tempo'].max()

print(f"Tempo before normalization:")
print(f"  Min: {tempo_min:.3f}")
print(f"  Max: {tempo_max:.3f}")
print(f"  Mean: {joined_filtered['Tempo'].mean():.3f}")

joined_filtered = joined_filtered.copy()
joined_filtered['Tempo'] = (joined_filtered['Tempo'] - tempo_min) / (tempo_max - tempo_min)

print(f"\nTempo after normalization:")
print(f"  Min: {joined_filtered['Tempo'].min():.3f}")
print(f"  Max: {joined_filtered['Tempo'].max():.3f}")
print(f"  Mean: {joined_filtered['Tempo'].mean():.3f}")

print(f"\nAll feature ranges after normalization:")
for feat in TASTE_FEATURES:
    print(f"  {feat:20s}  [{joined_filtered[feat].min():.3f}, {joined_filtered[feat].max():.3f}]")

In [ ]:
print("=" * 60)
print("STEP 5: BUILD TASTE PROFILES (mean per user)")
print("=" * 60)

taste_profiles = joined_filtered.groupby('pid')[TASTE_FEATURES].mean()

print(f"Taste profiles built: {len(taste_profiles)} users")
print(f"Features per profile: {len(TASTE_FEATURES)}")
print(f"\nTaste profile statistics across all users:")
print(taste_profiles.describe().round(4))

print(f"\nExample profiles:")
examples = taste_profiles.head(5)
for pid, row in examples.iterrows():
    n_songs = songs_per_playlist[pid]
    print(f"\n  User (pid={pid}), {n_songs} songs:")
    for feat in TASTE_FEATURES:
        bar = '#' * int(row[feat] * 30)
        print(f"    {feat:20s}  {row[feat]:.3f}  {bar}")

In [ ]:
print("=" * 60)
print("STEP 6: SAVE")
print("=" * 60)

# Save taste profiles
taste_profiles.to_csv('outputs/taste_profiles.csv')
print(f"Saved taste_profiles.csv ({len(taste_profiles)} users x {len(TASTE_FEATURES)} features)")

# Per-user song list for downstream reward-function validation in 03_parameter_tuning.ipynb
user_liked_songs = joined_filtered[['pid', 'artist_name', 'track_name']].reset_index(drop=True)
user_liked_songs.to_csv('outputs/user_liked_songs.csv', index=False)
print(f"Saved user_liked_songs.csv ({len(user_liked_songs)} rows, {user_liked_songs['pid'].nunique()} users)")


print(f"\nTempo normalization params (apply same scaling to candidate songs):")
print(f"  min={tempo_min:.3f}, max={tempo_max:.3f}")

print(f"\nFinal output:")
print(taste_profiles.head(10).round(4))